# 04 - Suplemen laporan: lingkungan, pengukuran waktu, statistik, cek kualitas, uji leakage

Notebook ini melengkapi laporan dengan angka yang **belum tercetak** oleh notebook 01-03:

1. Lingkungan (versi Python/Spark/Java, CPU/RAM), schema, jumlah partisi, contoh data, ukuran file.
2. Statistik deskriptif kolom numerik.
3. Pengukuran waktu 3 operasi dengan metode yang sama (3 kali jalan per operasi).
4. Pemeriksaan kualitas **setelah** transformasi (rentang nilai + schema pada `modeling_table`).
5. Uji sensitivitas *leakage* `SCOREMARGIN` (tanpa fitur itu, dan dengan margin sebelum tembakan).

Prasyarat: notebook 01-03 sudah dijalankan (folder `output/parquet/` ada). Semua hasil juga disimpan ke
`output/supplement_*.csv|json` supaya bisa disalin ke laporan (sel berwarna kuning di laporan).

## Setup

In [1]:
import os, sys, json, time, platform, glob, statistics

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.environ.setdefault("JAVA_HOME", r"C:\Program Files\Java\jre-1.8")
os.environ.setdefault("HADOOP_HOME", r"C:\hadoop")
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]
# Driver memory harus diset sebelum JVM start (lihat README).
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 6g pyspark-shell"

DATASET_DIR = os.path.join(PROJECT_ROOT, "dataset")
PARQUET_DIR = os.path.join(PROJECT_ROOT, "output", "parquet")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "output")
SEED = 42

In [2]:
from pyspark.sql import SparkSession, functions as F, Window

spark = (
    SparkSession.builder.appName("NBA-Supplement")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 1. Lingkungan, schema, partisi, contoh data, ukuran file

In [3]:
env = {
    "python": sys.version.split()[0],
    "pyspark": spark.version,
    "java": spark._jvm.System.getProperty("java.version"),
    "os": platform.platform(),
    "cpu_logical_cores": os.cpu_count(),
    "spark_master": spark.sparkContext.master,
    "default_parallelism": spark.sparkContext.defaultParallelism,
    "shuffle_partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "driver_memory_arg": os.environ["PYSPARK_SUBMIT_ARGS"],
}
try:
    import psutil
    env["ram_total_gb"] = round(psutil.virtual_memory().total / 1024**3, 1)
except ImportError:
    env["ram_total_gb"] = "psutil tidak terpasang: baca dari Task Manager / pip install psutil"
print(json.dumps(env, indent=2))
with open(os.path.join(OUTPUT_DIR, "supplement_environment.json"), "w", encoding="utf-8") as f:
    json.dump(env, f, indent=2)

{
  "python": "3.12.10",
  "pyspark": "3.5.3",
  "java": "17.0.21",
  "os": "Windows-11-10.0.26200-SP0",
  "cpu_logical_cores": 12,
  "spark_master": "local[*]",
  "default_parallelism": 12,
  "shuffle_partitions": "16",
  "driver_memory_arg": "--driver-memory 6g pyspark-shell",
  "ram_total_gb": 15.9
}


In [4]:
nbastats = spark.read.parquet(os.path.join(PARQUET_DIR, "nbastats"))
shotdetail = spark.read.parquet(os.path.join(PARQUET_DIR, "shotdetail"))
modeling = spark.read.parquet(os.path.join(PARQUET_DIR, "modeling_table"))

for name, df in [("nbastats", nbastats), ("shotdetail", shotdetail), ("modeling_table", modeling)]:
    print(f"=== {name} | partisi (getNumPartitions) = {df.rdd.getNumPartitions()} ===")
    df.printSchema()

# Contoh data aman: hanya kolom statistik pertandingan (nama pemain publik), 3 baris.
shotdetail.select("GAME_ID", "GAME_EVENT_ID", "PERIOD", "PLAYER_NAME", "SHOT_TYPE",
                  "SHOT_ZONE_BASIC", "SHOT_DISTANCE", "LOC_X", "LOC_Y", "SHOT_MADE_FLAG", "season").show(3, truncate=False)

=== nbastats | partisi (getNumPartitions) = 16 ===
root
 |-- GAME_ID: integer (nullable = true)
 |-- EVENTNUM: integer (nullable = true)
 |-- EVENTMSGTYPE: integer (nullable = true)
 |-- EVENTMSGACTIONTYPE: integer (nullable = true)
 |-- PERIOD: integer (nullable = true)
 |-- WCTIMESTRING: string (nullable = true)
 |-- PCTIMESTRING: timestamp (nullable = true)
 |-- HOMEDESCRIPTION: string (nullable = true)
 |-- NEUTRALDESCRIPTION: string (nullable = true)
 |-- VISITORDESCRIPTION: string (nullable = true)
 |-- SCORE: string (nullable = true)
 |-- SCOREMARGIN: string (nullable = true)
 |-- PERSON1TYPE: integer (nullable = true)
 |-- PLAYER1_ID: integer (nullable = true)
 |-- PLAYER1_NAME: string (nullable = true)
 |-- PLAYER1_TEAM_ID: integer (nullable = true)
 |-- PLAYER1_TEAM_CITY: string (nullable = true)
 |-- PLAYER1_TEAM_NICKNAME: string (nullable = true)
 |-- PLAYER1_TEAM_ABBREVIATION: string (nullable = true)
 |-- PERSON2TYPE: integer (nullable = true)
 |-- PLAYER2_ID: integer (nu

In [5]:
def dir_size_gb(paths):
    return round(sum(os.path.getsize(p) for p in paths) / 1024**3, 3)

sizes = {
    "csv_nbastats_gb": dir_size_gb(glob.glob(os.path.join(DATASET_DIR, "nbastats_*.csv"))),
    "csv_shotdetail_gb": dir_size_gb(glob.glob(os.path.join(DATASET_DIR, "shotdetail_*.csv"))),
    "csv_all_486_files_gb": dir_size_gb(glob.glob(os.path.join(DATASET_DIR, "*.csv"))),
}
sizes["csv_two_core_families_gb"] = round(sizes["csv_nbastats_gb"] + sizes["csv_shotdetail_gb"], 3)
for fam in ["nbastats", "shotdetail", "modeling_table"]:
    files = [os.path.join(dp, fn) for dp, _, fns in os.walk(os.path.join(PARQUET_DIR, fam)) for fn in fns]
    sizes[f"parquet_{fam}_gb"] = dir_size_gb(files)
print(json.dumps(sizes, indent=2))
with open(os.path.join(OUTPUT_DIR, "supplement_sizes.json"), "w", encoding="utf-8") as f:
    json.dump(sizes, f, indent=2)

{
  "csv_nbastats_gb": 2.699,
  "csv_shotdetail_gb": 1.113,
  "csv_all_486_files_gb": 3.813,
  "csv_two_core_families_gb": 3.812,
  "parquet_nbastats_gb": 0.406,
  "parquet_shotdetail_gb": 0.06,
  "parquet_modeling_table_gb": 0.025
}


## 2. Statistik deskriptif

In [6]:
desc_cols = ["SHOT_DISTANCE", "LOC_X", "LOC_Y", "PERIOD", "MINUTES_REMAINING", "SECONDS_REMAINING", "SHOT_MADE_FLAG"]
desc = shotdetail.select(*desc_cols).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
desc.show(truncate=False)
desc.toPandas().to_csv(os.path.join(OUTPUT_DIR, "supplement_describe_shotdetail.csv"), index=False)

desc_model = modeling.select("time_remaining_seconds", "SCOREMARGIN", "SHOT_DISTANCE").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
desc_model.show(truncate=False)
desc_model.toPandas().to_csv(os.path.join(OUTPUT_DIR, "supplement_describe_modeling.csv"), index=False)

+-------+------------------+-------------------+-----------------+------------------+-----------------+------------------+-------------------+
|summary|SHOT_DISTANCE     |LOC_X              |LOC_Y            |PERIOD            |MINUTES_REMAINING|SECONDS_REMAINING |SHOT_MADE_FLAG     |
+-------+------------------+-------------------+-----------------+------------------+-----------------+------------------+-------------------+
|count  |6094438           |6094438            |6094438          |6094438           |6094438          |6094438           |6094438            |
|mean   |12.346868078730147|-1.4803634067653162|79.6260157212199 |2.478029803568434 |5.340837169891629|28.758833218091645|0.4549707454567591 |
|stddev |10.011477252707413|110.74965624349635 |88.83545847437173|1.1378795770008192|3.466402372418305|17.447863129128894|0.49796827903350754|
|min    |0                 |-250               |-52              |1                 |0                |0                 |0                  |

## 3. Pengukuran waktu (metode seragam)

Metode: untuk setiap operasi, bangun DataFrame lazy lalu picu eksekusi penuh dengan sink `noop`
(`write.format("noop")`) - plan dijalankan tanpa biaya menulis hasil. Waktu = `time.perf_counter()` (wall-clock)
di sekitar aksi. Tiap operasi dijalankan 3 kali; laporan mencantumkan run-1 (cold) dan median 3 run.
Tidak ada `cache()` di notebook ini dan `spark.catalog.clearCache()` dipanggil sebelum pengukuran.

In [7]:
spark.catalog.clearCache()
shotdetail.createOrReplaceTempView("shotdetail_v")

def op_a_api():
    return (shotdetail.withColumn("is_3pt", (F.col("SHOT_TYPE") == "3PT Field Goal").cast("int"))
            .groupBy("season").agg(F.sum("is_3pt").alias("n3"), F.count("*").alias("n"))
            .withColumn("rate", F.col("n3") / F.col("n")))

def op_b_sql():
    return spark.sql("""
        SELECT SHOT_ZONE_BASIC, SHOT_ZONE_AREA,
               SUM(SHOT_MADE_FLAG) / SUM(SHOT_ATTEMPTED_FLAG) AS fg_pct
        FROM shotdetail_v GROUP BY SHOT_ZONE_BASIC, SHOT_ZONE_AREA""")

def op_c_join_window():
    fg = nbastats.filter(F.col("EVENTMSGTYPE").isin(1, 2))
    w_dedup = Window.partitionBy("GAME_ID", "EVENTNUM").orderBy(F.monotonically_increasing_id())
    fg = fg.withColumn("_rn", F.row_number().over(w_dedup)).filter("_rn = 1").drop("_rn")
    margin = F.when(F.col("SCOREMARGIN") == "TIE", F.lit(0)).otherwise(
        F.regexp_replace(F.col("SCOREMARGIN"), "[^0-9-]", "").cast("int"))
    w_ff = Window.partitionBy("GAME_ID").orderBy("EVENTNUM").rowsBetween(Window.unboundedPreceding, 0)
    fg = fg.withColumn("margin_ffill", F.last(margin, ignorenulls=True).over(w_ff))
    joined = fg.alias("n").join(
        shotdetail.alias("s"), (F.col("n.GAME_ID") == F.col("s.GAME_ID")) & (F.col("n.EVENTNUM") == F.col("s.GAME_EVENT_ID")), "inner")
    return joined.select(F.col("n.GAME_ID").alias("GAME_ID"), F.col("n.EVENTNUM").alias("EVENTNUM"),
                         F.col("n.margin_ffill").alias("margin_ffill"), F.col("s.SHOT_MADE_FLAG").alias("SHOT_MADE_FLAG"))

OPS = [
    ("A. groupBy/agregasi 3PT-rate per musim (DataFrame API)", op_a_api),
    ("B. FG% per zona (Spark SQL)", op_b_sql),
    ("C. dedup + forward-fill (window) + join nbastats x shotdetail", op_c_join_window),
]

def time_once(builder):
    df = builder()
    t0 = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    return time.perf_counter() - t0

rows = []
for name, builder in OPS:
    runs = [time_once(builder) for _ in range(3)]
    rows.append({"operasi": name, "run1_cold_s": round(runs[0], 2), "run2_s": round(runs[1], 2),
                 "run3_s": round(runs[2], 2), "median_s": round(statistics.median(runs), 2)})
    print(rows[-1])

import pandas as pd
timing = pd.DataFrame(rows)
timing.to_csv(os.path.join(OUTPUT_DIR, "supplement_timing.csv"), index=False)
timing

{'operasi': 'A. groupBy/agregasi 3PT-rate per musim (DataFrame API)', 'run1_cold_s': 0.88, 'run2_s': 0.36, 'run3_s': 0.33, 'median_s': 0.36}
{'operasi': 'B. FG% per zona (Spark SQL)', 'run1_cold_s': 0.96, 'run2_s': 0.67, 'run3_s': 0.53, 'median_s': 0.67}
{'operasi': 'C. dedup + forward-fill (window) + join nbastats x shotdetail', 'run1_cold_s': 6.86, 'run2_s': 5.2, 'run3_s': 4.92, 'median_s': 5.2}


,operasi,run1_cold_s,run2_s,run3_s,median_s
0,A. groupBy/agregasi 3PT-rate per musim (DataFr...,0.88,0.36,0.33,0.36
1,B. FG% per zona (Spark SQL),0.96,0.67,0.53,0.67
2,C. dedup + forward-fill (window) + join nbasta...,6.86,5.20,4.92,5.20


## 4. Pemeriksaan kualitas SETELAH transformasi (`modeling_table`)

Memeriksa rentang nilai, domain label, dan schema pada tabel final yang masuk ke model.

In [8]:
EXPECTED_SCHEMA = {
    "SHOT_DISTANCE": "int", "SHOT_ZONE_BASIC": "string", "SHOT_ZONE_AREA": "string", "SHOT_TYPE": "string",
    "PERIOD": "int", "time_remaining_seconds": "int", "SCOREMARGIN": "int", "SEASON": "int", "SHOT_MADE_FLAG": "int",
}
actual_schema = {f.name: f.dataType.simpleString() for f in modeling.schema.fields}

agg = modeling.agg(
    F.count("*").alias("n"),
    F.min("SHOT_DISTANCE").alias("dist_min"), F.max("SHOT_DISTANCE").alias("dist_max"),
    F.min("PERIOD").alias("period_min"), F.max("PERIOD").alias("period_max"),
    F.min("time_remaining_seconds").alias("t_min"), F.max("time_remaining_seconds").alias("t_max"),
    F.min("SEASON").alias("season_min"), F.max("SEASON").alias("season_max"),
).collect()[0].asDict()
labels = sorted(r[0] for r in modeling.select("SHOT_MADE_FLAG").distinct().collect())

checks = {
    "schema_sesuai": actual_schema == EXPECTED_SCHEMA,
    "shot_distance_non_negatif": agg["dist_min"] >= 0,
    "period_1_sampai_8": 1 <= agg["period_min"] and agg["period_max"] <= 8,
    "time_remaining_0_sampai_720": 0 <= agg["t_min"] and agg["t_max"] <= 720,
    "season_1996_sampai_2024": agg["season_min"] >= 1996 and agg["season_max"] <= 2024,
    "label_hanya_0_1": labels == [0, 1],
}
print("schema aktual:", actual_schema)
print("ringkasan rentang:", agg)
for k, v in checks.items():
    print(("PASS " if v else "FAIL ") + k)
with open(os.path.join(OUTPUT_DIR, "supplement_post_transform_checks.json"), "w", encoding="utf-8") as f:
    json.dump({"schema": actual_schema, "ranges": agg, "checks": checks}, f, indent=2, default=str)

schema aktual: {'SHOT_DISTANCE': 'int', 'SHOT_ZONE_BASIC': 'string', 'SHOT_ZONE_AREA': 'string', 'SHOT_TYPE': 'string', 'PERIOD': 'int', 'time_remaining_seconds': 'int', 'SCOREMARGIN': 'int', 'SEASON': 'int', 'SHOT_MADE_FLAG': 'int'}
ringkasan rentang: {'n': 6049202, 'dist_min': 0, 'dist_max': 90, 'period_min': 1, 'period_max': 8, 't_min': 0, 't_max': 720, 'season_min': 1996, 'season_max': 2024}
PASS schema_sesuai
PASS shot_distance_non_negatif
PASS period_1_sampai_8
PASS time_remaining_0_sampai_720
PASS season_1996_sampai_2024
PASS label_hanya_0_1


## 5. Uji sensitivitas leakage `SCOREMARGIN`

Di notebook 02-03, forward-fill `SCOREMARGIN` memakai window `rowsBetween(unboundedPreceding, 0)`, yaitu
**termasuk baris tembakan itu sendiri**. Pada tembakan yang masuk, `SCOREMARGIN` di `nbastats` sudah memuat
poin tembakan tersebut; pada tembakan yang meleset, nilainya masih margin sebelumnya. Ini berpotensi membocorkan
label. Tiga varian dibandingkan dengan split acak 80/20 (seed 42), model dan hyperparameter sama seperti notebook 03:

- **V0** - replikasi notebook 03 (margin termasuk baris sendiri).
- **V1** - tanpa fitur `SCOREMARGIN`.
- **V2** - margin *sebelum* tembakan (window berakhir di baris sebelumnya, `-1`).

In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

CATEGORICAL = ["SHOT_ZONE_BASIC", "SHOT_ZONE_AREA", "SHOT_TYPE"]

def build_table(frame_end):
    """Rebuild modeling table seperti notebook 03; frame_end=0 (V0) atau -1 (V2)."""
    nb = nbastats.filter(F.col("EVENTMSGTYPE").isin(1, 2))
    w_dedup = Window.partitionBy("GAME_ID", "EVENTNUM").orderBy(F.monotonically_increasing_id())
    nb = nb.withColumn("_rn", F.row_number().over(w_dedup)).filter("_rn = 1").drop("_rn")
    margin = F.when(F.col("SCOREMARGIN") == "TIE", F.lit(0)).otherwise(
        F.regexp_replace(F.col("SCOREMARGIN"), "[^0-9-]", "").cast("int"))
    w_ff = Window.partitionBy("GAME_ID").orderBy("EVENTNUM").rowsBetween(Window.unboundedPreceding, frame_end)
    nb = nb.withColumn("margin_ffill", F.last(margin, ignorenulls=True).over(w_ff))
    w_sd = Window.partitionBy("GAME_ID", "GAME_EVENT_ID").orderBy(F.monotonically_increasing_id())
    sd = shotdetail.withColumn("_rn", F.row_number().over(w_sd)).filter("_rn = 1").drop("_rn")
    joined = nb.alias("n").join(
        sd.alias("s"), (F.col("n.GAME_ID") == F.col("s.GAME_ID")) & (F.col("n.EVENTNUM") == F.col("s.GAME_EVENT_ID")), "inner")
    out = joined.select(
        F.col("n.PERIOD").alias("PERIOD"), F.col("n.margin_ffill").alias("SCOREMARGIN"),
        F.col("s.season").alias("SEASON"), F.col("s.SHOT_DISTANCE").alias("SHOT_DISTANCE"),
        F.col("s.SHOT_ZONE_BASIC").alias("SHOT_ZONE_BASIC"), F.col("s.SHOT_ZONE_AREA").alias("SHOT_ZONE_AREA"),
        F.col("s.SHOT_TYPE").alias("SHOT_TYPE"),
        (F.col("s.MINUTES_REMAINING") * 60 + F.col("s.SECONDS_REMAINING")).alias("time_remaining_seconds"),
        F.col("s.SHOT_MADE_FLAG").alias("SHOT_MADE_FLAG"),
    )
    return out.dropna()

def fit_eval(table, numeric_cols, name):
    t = table
    for c in numeric_cols:
        t = t.withColumn(c, F.col(c).cast("double"))
    t = t.withColumn("label", F.col("SHOT_MADE_FLAG").cast("double"))
    train, test = t.randomSplit([0.8, 0.2], seed=SEED)
    stages = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in CATEGORICAL]
    stages.append(OneHotEncoder(inputCols=[c + "_idx" for c in CATEGORICAL], outputCols=[c + "_ohe" for c in CATEGORICAL]))
    stages.append(VectorAssembler(inputCols=[c + "_ohe" for c in CATEGORICAL] + numeric_cols, outputCol="features"))
    stages.append(LogisticRegression(featuresCol="features", labelCol="label", maxIter=50))
    pred = Pipeline(stages=stages).fit(train).transform(test)
    auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                        metricName="areaUnderROC").evaluate(pred)
    acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                            metricName="accuracy").evaluate(pred)
    res = {"varian": name, "train_rows": train.count(), "test_rows": test.count(),
           "accuracy": round(acc, 4), "auc_roc": round(auc, 4)}
    print(res)
    return res

NUM_FULL = ["SHOT_DISTANCE", "PERIOD", "time_remaining_seconds", "SCOREMARGIN", "SEASON"]
NUM_NO_MARGIN = ["SHOT_DISTANCE", "PERIOD", "time_remaining_seconds", "SEASON"]

t_v0 = build_table(0)
results = [
    fit_eval(t_v0, NUM_FULL, "V0 margin termasuk baris sendiri (notebook 03)"),
    fit_eval(t_v0, NUM_NO_MARGIN, "V1 tanpa SCOREMARGIN"),
    fit_eval(build_table(-1), NUM_FULL, "V2 margin sebelum tembakan (window -1)"),
]
pd.DataFrame(results).to_csv(os.path.join(OUTPUT_DIR, "supplement_leakage_ablation.csv"), index=False)
pd.DataFrame(results)

{'varian': 'V0 margin termasuk baris sendiri (notebook 03)', 'train_rows': 4840493, 'test_rows': 1208709, 'accuracy': 0.6138, 'auc_roc': 0.6193}
{'varian': 'V1 tanpa SCOREMARGIN', 'train_rows': 4840493, 'test_rows': 1208709, 'accuracy': 0.6138, 'auc_roc': 0.6193}
{'varian': 'V2 margin sebelum tembakan (window -1)', 'train_rows': 4811212, 'test_rows': 1201323, 'accuracy': 0.6153, 'auc_roc': 0.6171}


,varian,train_rows,test_rows,accuracy,auc_roc
0,V0 margin termasuk baris sendiri (notebook 03),4840493,1208709,0.6138,0.6193
1,V1 tanpa SCOREMARGIN,4840493,1208709,0.6138,0.6193
2,V2 margin sebelum tembakan (window -1),4811212,1201323,0.6153,0.6171


In [10]:
spark.stop()